# Decision Tree Feature Importance Comparison


In [1]:
import pandas as pd
from scipy.stats import spearmanr
import itertools

In [ ]:
strat_A_csv_path = "../RESULTS/FEATURES/decision_tree_oversampling.csv"
strat_B_csv_path = "../RESULTS/FEATURES/decision_tree_undersampling.csv"

df_strat_A = pd.read_csv(strat_A_csv_path).sort_values(by=['method', 'rank'])
df_strat_B = pd.read_csv(strat_B_csv_path).sort_values(by=['method', 'rank'])


def calculate_comparison_metrics(df_1, df_2, penalty_rank=11):
    """Calculates Overlap, Jaccard, and Spearman between two top-10 result dataframes."""
    dict_1 = dict(zip(df_1['feature'], df_1['rank']))
    dict_2 = dict(zip(df_2['feature'], df_2['rank']))
    
    set_1 = set(dict_1.keys())
    set_2 = set(dict_2.keys())
    
    # 1. Top-10 Overlap
    overlap = len(set_1.intersection(set_2))
    
    # 2. Jaccard Similarity
    union = set_1.union(set_2)
    jaccard = overlap / len(union) if len(union) > 0 else 0
    
    # 3. Spearman Rank Correlation
    ranks_1 = []
    ranks_2 = []
    for feature in union:
        # Assign rank 11 if the feature fell out of the top 10 in one of the lists
        ranks_1.append(dict_1.get(feature, penalty_rank))
        ranks_2.append(dict_2.get(feature, penalty_rank))
        
    spearman_corr, _ = spearmanr(ranks_1, ranks_2)
    
    return overlap, jaccard, spearman_corr

results = []

# Dynamically get the XAI methods present in data
xai_methods = df_strat_A['method'].unique()

# RQ3: Effect of Preprocessing Strategy
# Comparing Strategy A vs Strategy B
print("Processing RQ3...")

for method in xai_methods:
    # Get the top 10 for this specific method from BOTH strategies
    df_a_method = df_strat_A[df_strat_A['method'] == method]
    df_b_method = df_strat_B[df_strat_B['method'] == method]
    
    if not df_a_method.empty and not df_b_method.empty:
        overlap, jaccard, spearman = calculate_comparison_metrics(df_a_method, df_b_method)
        results.append({
            "Research_Question": "RQ3",
            "Model": "Decision Tree", 
            "Comparison": "SMOTE vs Random Undersampling",
            "Fixed_Variable": f"Method: {method}",
            "Top_10_Overlap": overlap,
            "Jaccard_Similarity": jaccard,
            "Spearman_Correlation": spearman
        })

# RQ4: Effect of Explainability Method
# Comparing Method vs Method
print("Processing RQ4...")

baseline_strategy = "SMOTE"

for method_1, method_2 in itertools.combinations(xai_methods, 2):
    
    # Extract both methods from the SAME Strategy A dataframe
    df_m1 = df_strat_A[df_strat_A['method'] == method_1]
    df_m2 = df_strat_A[df_strat_A['method'] == method_2]
    
    if not df_m1.empty and not df_m2.empty:
        overlap, jaccard, spearman = calculate_comparison_metrics(df_m1, df_m2)
        results.append({
            "Research_Question": "RQ4",
            "Model": "Decision Tree",
            "Comparison": f"{method_1} vs {method_2}",
            "Fixed_Variable": f"Strategy: {baseline_strategy}",
            "Top_10_Overlap": overlap,
            "Jaccard_Similarity": jaccard,
            "Spearman_Correlation": spearman
        })

results_df = pd.DataFrame(results)

display(results_df)


EmptyDataError: No columns to parse from file